### Bank Marketing — Logistic Regression riêng
Notebook triển khai độc lập thuật toán Logistic Regression, theo bố cục của notebook KNN và Decision Tree trong đồ án.
- Đặt notebook này trong thư mục Đồ án cuối kỳ/, cùng cấp với thư mục data/.
- Dữ liệu đầu vào: data/bank-additional-full.csv.
- Giữ cách tiền xử lý và chia dữ liệu thống nhất với file Main: loại bản ghi trùng/thiếu nhãn, bỏ duration, chia train/test 80/20 với stratify và random_state=42.
- Các phần thực nghiệm: 4. Logistic Regression baseline, 9.1. SMOTE và chỉnh ngưỡng, 10.1. Tinh chỉnh siêu tham số.
- Không dùng tập test để chọn ngưỡng hoặc siêu tham số. Các quyết định này được thực hiện bằng train/CV.

In [ ]:
%pip install pandas numpy matplotlib scikit-learn imbalanced-learn

### Chuẩn bị dữ liệu

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

# Dùng 3-fold để chạy nhanh; có thể đổi thành 5 khi cần kiểm tra kỹ hơn.
N_FOLDS = 3
RANDOM_STATE = 42

ImportError: cannot import name 'BackendFilter' from 'matplotlib.backends.registry' (c:\Users\ducat\AppData\Local\Programs\Python\Python310\lib\site-packages\matplotlib\backends\registry.py)

In [ ]:
# Đọc dữ liệu. Notebook cần nằm cạnh thư mục data/.
DATA_PATH = Path("data/bank-additional-full.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_PATH}. Hãy đặt notebook trong thư mục Đồ án cuối kỳ/ "
        "hoặc sửa DATA_PATH cho đúng vị trí file CSV."
    )

df = pd.read_csv(DATA_PATH, sep=";")
print("Kích thước dữ liệu:", df.shape)
display(df.head())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Loại bản ghi trùng và bản ghi thiếu nhãn; giữ "unknown" như một nhóm riêng.
df_model = df.drop_duplicates().dropna(subset=["y"]).copy()

# duration chỉ biết sau khi cuộc gọi kết thúc nên không dùng để dự đoán trước cuộc gọi.
X = df_model.drop(columns=["y", "duration"])
y = df_model["y"].map({"no": 0, "yes": 1})

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# Chỉ fit bộ tiền xử lý trên train để tránh rò rỉ thông tin từ test.
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out()

print(f"Số dòng sau khi loại trùng/thiếu nhãn: {len(df_model):,}")
print(f"Số đặc trưng sau mã hóa: {len(feature_names)}")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print("\nPhân bố nhãn trong train:")
print(y_train.map({0: "no", 1: "yes"}).value_counts())
print("\nTỷ lệ nhãn trong train (%):")
print(y_train.map({0: "no", 1: "yes"}).value_counts(normalize=True).mul(100).round(2))

### 4. Thuật toán Logistic Regression
#### 4.1. Giới thiệu thuật toán
Logistic Regression (Hồi quy Logistic) là thuật toán học có giám sát thường dùng cho bài toán phân loại. Mô hình tính tổng có trọng số của các đặc trưng, sau đó đưa kết quả qua hàm Sigmoid để chuyển thành xác suất thuộc lớp yes:
\[
P(y=1\mid X)=\frac{1}{1+e^{-z}},\qquad z=b+w_1x_1+\cdots+w_nx_n
\]
Với bài toán Bank Marketing, nếu xác suất dự đoán đạt ngưỡng 0,5 trở lên thì mô hình dự đoán khách hàng đăng ký tiền gửi kỳ hạn (yes), ngược lại là no.
- Ưu điểm: huấn luyện tương đối nhanh, dễ giải thích thông qua hệ số, có thể trả về xác suất dự đoán.
- Nhược điểm: ranh giới phân loại cơ bản là tuyến tính; nhạy với thang đo nên cần chuẩn hóa biến số. Kết quả cũng chịu ảnh hưởng bởi mất cân bằng lớp.
Cấu hình baseline dùng max_iter=2000 để hỗ trợ hội tụ và class_weight="balanced" để tăng trọng số cho lớp yes đang chiếm tỷ lệ nhỏ. random_state=42 giúp thực nghiệm có thể lặp lại.

#### 4.2. Huấn luyện và dự đoán

In [ ]:
from sklearn.linear_model import LogisticRegression

# Khởi tạo mô hình Logistic Regression baseline.
model_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

# Huấn luyện trên tập train.
model_lr.fit(X_train, y_train)

# Lấy đúng cột xác suất ứng với lớp 1 (yes).
yes_index_lr = list(model_lr.classes_).index(1)
y_pred_lr = model_lr.predict(X_test)
y_proba_lr = model_lr.predict_proba(X_test)[:, yes_index_lr]

print("Huấn luyện và dự đoán Logistic Regression hoàn tất!")
print("\n10 nhãn dự đoán đầu tiên:", y_pred_lr[:10])
print("10 xác suất dự đoán đầu tiên:", y_proba_lr[:10].round(4))

#### 4.3. Đánh giá mô hình

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

# Tính các chỉ số cho Logistic Regression baseline.
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, zero_division=0)
recall_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)
auc_lr = roc_auc_score(y_test, y_proba_lr)

print("===== LOGISTIC REGRESSION BASELINE =====")
print(f"Accuracy : {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall   : {recall_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"ROC-AUC  : {auc_lr:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    y_test,
    y_pred_lr,
    labels=[0, 1],
    target_names=["No", "Yes"],
    zero_division=0,
))

#### 4.4. Trực quan hóa kết quả

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_lr,
    labels=[0, 1],
    display_labels=["No", "Yes"],
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("Logistic Regression - Confusion Matrix")

RocCurveDisplay.from_predictions(
    y_test,
    y_proba_lr,
    name="Logistic Regression",
    ax=axes[1],
)
axes[1].plot([0, 1], [0, 1], "k--", label="Random baseline")
axes[1].set_title("Logistic Regression - ROC Curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

#### 4.5. Nhận xét và lưu kết quả

In [ ]:
# Lưu kết quả vào danh sách chung để có thể tổng hợp với KNN/Decision Tree.
if "all_results" not in globals():
    all_results = []
if "all_probas" not in globals():
    all_probas = {}

# Xóa phiên bản cũ nếu chạy lại cell, tránh tạo dòng trùng.
all_results[:] = [
    row for row in all_results
    if row.get("Model") != "Logistic Regression (baseline)"
]
all_results.append({
    "Model": "Logistic Regression (baseline)",
    "Accuracy": accuracy_lr,
    "Precision": precision_lr,
    "Recall": recall_lr,
    "F1-score": f1_lr,
    "ROC-AUC": auc_lr,
})
all_probas["Logistic Regression (baseline)"] = y_proba_lr

print("Đã lưu kết quả Logistic Regression baseline!")

### 9.1. Logistic Regression — Xử lý mất cân bằng và chỉnh ngưỡng

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)


def lr_metrics(y_true, y_pred, y_proba):
    """Tính các độ đo; lớp yes = 1 là lớp dương."""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
    }


def lr_save_result(name, metrics, proba):
    """Lưu/cập nhật kết quả để cell chạy lại không tạo bản ghi trùng."""
    global all_results, all_probas

    if "all_results" not in globals():
        all_results = []
    if "all_probas" not in globals():
        all_probas = {}

    all_results[:] = [row for row in all_results if row.get("Model") != name]
    all_results.append({"Model": name, **metrics})
    all_probas[name] = proba


def lr_report(name, y_pred, y_proba):
    """In các độ đo, báo cáo phân loại và vẽ hai biểu đồ."""
    metrics = lr_metrics(y_test, y_pred, y_proba)

    print(f"===== {name.upper()} =====")
    for metric, value in metrics.items():
        print(f"{metric:<10}: {value:.4f}")

    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["No", "Yes"],
        zero_division=0,
    ))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=[0, 1],
        display_labels=["No", "Yes"],
        cmap="Blues",
        ax=axes[0],
    )
    axes[0].set_title(f"{name} - Confusion Matrix")

    RocCurveDisplay.from_predictions(
        y_test,
        y_proba,
        name=name,
        ax=axes[1],
    )
    axes[1].plot([0, 1], [0, 1], "k--", label="Random baseline")
    axes[1].set_title(f"{name} - ROC Curve")
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    plt.show()

    lr_save_result(name, metrics, y_proba)
    return metrics

#### 9.1.1. Cân bằng dữ liệu bằng SMOTE
SMOTE tạo thêm mẫu thuộc lớp thiểu số dựa trên các láng giềng gần. Chỉ áp dụng trên tập train; tập test giữ nguyên phân bố ban đầu.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

print("Phân bố lớp trước SMOTE:")
print(y_train.value_counts().sort_index())

# Chỉ resample tập train; tuyệt đối không resample tập test.
smote_lr = SMOTE(random_state=RANDOM_STATE)
X_train_smote_lr, y_train_smote_lr = smote_lr.fit_resample(X_train, y_train)

print("\nPhân bố lớp sau SMOTE:")
print(pd.Series(y_train_smote_lr).value_counts().sort_index())

# Đặt class_weight=None để đánh giá riêng tác động của SMOTE.
model_lr_smote = LogisticRegression(
    max_iter=2000,
    class_weight=None,
    random_state=RANDOM_STATE,
)
model_lr_smote.fit(X_train_smote_lr, y_train_smote_lr)

yes_index_lr_smote = list(model_lr_smote.classes_).index(1)
y_pred_lr_smote = model_lr_smote.predict(X_test)
y_proba_lr_smote = model_lr_smote.predict_proba(X_test)[:, yes_index_lr_smote]

metrics_lr_smote = lr_report(
    "Logistic Regression (SMOTE)",
    y_pred_lr_smote,
    y_proba_lr_smote,
)

#### 9.1.2. Chỉnh ngưỡng phân loại
Mặc định, Logistic Regression dự đoán yes khi xác suất từ 0,5 trở lên. Với dữ liệu mất cân bằng, ngưỡng 0,5 chưa chắc tối ưu cho F1-score.

Ta dùng cross_val_predict để lấy xác suất OOF trên tập train: mỗi dòng được dự đoán bởi mô hình chưa học dòng đó. Sau đó chọn ngưỡng cho F1-score cao nhất. Không dùng nhãn của tập test để lựa chọn ngưỡng.

In [ ]:
print(f"Đang chọn ngưỡng bằng {N_FOLDS}-fold trên train...", flush=True)

cv_lr_threshold = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Pipeline giúp mỗi fold tự fit bộ tiền xử lý trên phần train của fold đó.
lr_threshold_pipeline = SkPipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        ("lr", clone(model_lr)),
    ]
)

oof_proba_lr = cross_val_predict(
    lr_threshold_pipeline,
    X_train_raw,
    y_train,
    cv=cv_lr_threshold,
    method="predict_proba",
    n_jobs=2,
)[:, 1]

precision_oof_lr, recall_oof_lr, thresholds_lr = precision_recall_curve(
    y_train,
    oof_proba_lr,
)

# Precision/Recall có nhiều hơn thresholds một phần tử.
f1_oof_lr = np.divide(
    2 * precision_oof_lr[:-1] * recall_oof_lr[:-1],
    precision_oof_lr[:-1] + recall_oof_lr[:-1],
    out=np.zeros_like(precision_oof_lr[:-1]),
    where=(precision_oof_lr[:-1] + recall_oof_lr[:-1]) > 0,
)

df_thr_lr = pd.DataFrame({
    "Threshold": thresholds_lr,
    "Precision": precision_oof_lr[:-1],
    "Recall": recall_oof_lr[:-1],
    "F1-score": f1_oof_lr,
})

best_idx_lr = int(df_thr_lr["F1-score"].idxmax())
best_threshold_lr = float(df_thr_lr.loc[best_idx_lr, "Threshold"])

print(f"Ngưỡng được chọn trên train: {best_threshold_lr:.4f}")
print(f"F1-score OOF tại ngưỡng được chọn: {df_thr_lr.loc[best_idx_lr, 'F1-score']:.4f}")
display(df_thr_lr.sort_values("F1-score", ascending=False).head(10).round(4))

# Áp dụng ngưỡng tìm được lên xác suất baseline của tập test.
y_pred_lr_thr = (y_proba_lr >= best_threshold_lr).astype(int)
metrics_lr_thr = lr_report(
    "Logistic Regression (threshold)",
    y_pred_lr_thr,
    y_proba_lr,
)

#### 9.1.3. So sánh các phương án

In [ ]:
# Tổng hợp kết quả trên cùng tập test.
metrics_lr_baseline = lr_metrics(y_test, y_pred_lr, y_proba_lr)
metrics_lr_smote = lr_metrics(y_test, y_pred_lr_smote, y_proba_lr_smote)
metrics_lr_thr = lr_metrics(y_test, y_pred_lr_thr, y_proba_lr)

df_lr_section9 = pd.DataFrame([
    {"Model": "Logistic Regression (baseline)", **metrics_lr_baseline},
    {"Model": "Logistic Regression (SMOTE)", **metrics_lr_smote},
    {"Model": "Logistic Regression (threshold)", **metrics_lr_thr},
]).set_index("Model")

display(df_lr_section9.round(4))

df_lr_section9[["Precision", "Recall", "F1-score", "ROC-AUC"]].plot(
    kind="bar",
    figsize=(12, 5),
    ylim=(0, 1),
    rot=10,
)
plt.title("Logistic Regression - So sánh baseline, SMOTE và threshold")
plt.xlabel("")
plt.ylabel("Giá trị")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

for model_name in ["Logistic Regression (SMOTE)", "Logistic Regression (threshold)"]:
    delta = df_lr_section9.loc[model_name] - df_lr_section9.loc["Logistic Regression (baseline)"]
    print(f"\n{model_name} so với baseline:")
    for metric in ["Precision", "Recall", "F1-score", "ROC-AUC"]:
        print(f"  Chênh lệch {metric}: {delta[metric]:+.4f}")

### 10.1. Logistic Regression — Tinh chỉnh siêu tham số

#### 10.1.1. Tìm kiếm siêu tham số

In [ ]:
import time
from sklearn.model_selection import GridSearchCV, StratifiedKFold

lr_pipeline_tuning = SkPipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        ("lr", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ]
)

param_grid_lr = {
    "lr__C": [0.01, 0.1, 1.0, 10.0],
    "lr__class_weight": [None, "balanced"],
}

cv_lr_tuning = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search_lr = GridSearchCV(
    estimator=lr_pipeline_tuning,
    param_grid=param_grid_lr,
    scoring={
        "f1": "f1",
        "recall": "recall",
        "roc_auc": "roc_auc",
    },
    refit="f1",
    cv=cv_lr_tuning,
    n_jobs=2,
    pre_dispatch=2,
    verbose=2,
    error_score="raise",
)

start_lr = time.perf_counter()
search_lr.fit(X_train_raw, y_train)
elapsed_lr = time.perf_counter() - start_lr

print(f"\nThời gian tìm kiếm: {elapsed_lr / 60:.2f} phút")
print("Tham số tốt nhất:", search_lr.best_params_)
print(f"F1-score CV tốt nhất: {search_lr.best_score_:.4f}")

cv_results_lr = pd.DataFrame(search_lr.cv_results_)
cols_lr = [
    "rank_test_f1",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_recall",
    "mean_test_roc_auc",
    "params",
]

display(
    cv_results_lr
    .sort_values("rank_test_f1")[cols_lr]
    .head(10)
    .round(4)
)

### Nhật ký các lần tìm kiếm trên train

In [ ]:
from datetime import datetime

output_dir = Path("ket_qua_logistic")
output_dir.mkdir(exist_ok=True)

# Cập nhật ghi chú trước mỗi lần thử nghiệm mới.
GHI_CHU_LR = "Khảo sát C = 0.01, 0.1, 1, 10; class_weight None/balanced"
run_id_lr = datetime.now().isoformat(timespec="seconds")
nhat_ky_lr = cv_results_lr[cols_lr].copy()
nhat_ky_lr.insert(0, "Thoi_diem", run_id_lr)
nhat_ky_lr.insert(1, "So_fold", N_FOLDS)
nhat_ky_lr["Ghi_chu"] = GHI_CHU_LR

history_file_lr = output_dir / "nhat_ky_cv.csv"
nhat_ky_lr.to_csv(
    history_file_lr,
    mode="a",
    header=not history_file_lr.exists(),
    index=False,
    encoding="utf-8-sig",
)

display(nhat_ky_lr.round(4))
print("Đã lưu nhật ký:", history_file_lr)

#### 10.1.2. Đánh giá mô hình sau tinh chỉnh

In [ ]:
# GridSearchCV đã fit lại cấu hình tốt nhất trên toàn bộ tập train.
model_lr_tuned = search_lr.best_estimator_

y_pred_lr_tuned = model_lr_tuned.predict(X_test_raw)
y_proba_lr_tuned = model_lr_tuned.predict_proba(X_test_raw)[:, 1]

metrics_lr_tuned = lr_report(
    "Logistic Regression (tuned)",
    y_pred_lr_tuned,
    y_proba_lr_tuned,
)

# So sánh bốn phiên bản trên cùng tập test.
df_lr_compare = pd.DataFrame([
    {"Model": "Logistic Regression (baseline)", **metrics_lr_baseline},
    {"Model": "Logistic Regression (SMOTE)", **metrics_lr_smote},
    {"Model": "Logistic Regression (threshold)", **metrics_lr_thr},
    {"Model": "Logistic Regression (tuned)", **metrics_lr_tuned},
]).set_index("Model")

display(df_lr_compare.round(4))

df_lr_compare[["Recall", "F1-score", "ROC-AUC"]].plot(
    kind="bar",
    figsize=(12, 5),
    ylim=(0, 1),
    rot=12,
)
plt.title("Logistic Regression - So sánh các phương án")
plt.xlabel("")
plt.ylabel("Giá trị")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTham số Logistic Regression tốt nhất:")
print(search_lr.best_params_)

print("\nChênh lệch tuned so với baseline:")
for metric in metrics_lr_baseline:
    difference = metrics_lr_tuned[metric] - metrics_lr_baseline[metric]
    print(f"{metric:<10}: {difference:+.4f}")

lr_save_result("Logistic Regression (tuned)", metrics_lr_tuned, y_proba_lr_tuned)

#### 10.1.3. Lưu kết quả thực nghiệm

In [ ]:
# Lưu bảng đánh giá test và bảng ngưỡng OOF để sử dụng trong báo cáo.
df_lr_compare.to_csv(output_dir / "so_sanh_test.csv", encoding="utf-8-sig")
df_thr_lr.to_csv(output_dir / "chon_nguong_oof.csv", index=False, encoding="utf-8-sig")

print("Đã lưu các tệp kết quả vào:", output_dir.resolve())
print("- so_sanh_test.csv")
print("- chon_nguong_oof.csv")
print("- nhat_ky_cv.csv")